In [1]:
import os
os.environ["R_HOME"] = "/homes/kbalzer/miniconda3/envs/qiime2-amplicon-2026.1/lib/R"

qiime_env = "/homes/kbalzer/miniconda3/envs/qiime2-amplicon-2026.1"
os.environ["PATH"] = f"{qiime_env}/bin:/usr/local/bin:/usr/bin:/bin"
os.environ["CONDA_PREFIX"] = qiime_env


In [2]:
import sys
import os

print(sys.executable)
print(sys.version)

/homes/kbalzer/miniconda3/envs/qiime2-amplicon-2026.1/bin/python
3.10.14 | packaged by conda-forge | (main, Mar 20 2024, 12:45:18) [GCC 12.3.0]


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ancombc2_heatmaps import (
    ANCOMBC2HeatmapPlotter,
    HeatmapConfig,
    MetadataConfig,
    ComparisonConfig,
    PathConfig,
    PlotTextConfig,
    HeatmapStyleConfig,
    TaxonomyConfig,
    SubsetSpec,
    TaxonTrajectoryPlotter,
    TrajectoryConfig,
    TrajectoryMetadataConfig,
    TrajectoryPathConfig,
    TrajectoryPlotConfig,
    TaxonBoxplotTrajectoryPlotter,
    PlotWorkflow,
)

In [4]:
PROJECT_DIR = Path("/vol/jlab/MicrobiomeAnalyses/Projects/JansenVanVuuren_Microbian2/Karl")

META_FP = PROJECT_DIR / "metadata_microbian2_26.02.2026.txt"

# Falls du die exports woanders hast, hier anpassen:
ANCOM_EXPORT_DIR = PROJECT_DIR / "real_ANCOMB_BC2"

OUTDIR = PROJECT_DIR / "notebook_plots_2026_1"
OUTDIR.mkdir(parents=True, exist_ok=True)

print("Metadata:", META_FP.exists(), META_FP)
print("ANCOM dir:", ANCOM_EXPORT_DIR.exists(), ANCOM_EXPORT_DIR)
print("Output dir:", OUTDIR)

Metadata: True /vol/jlab/MicrobiomeAnalyses/Projects/JansenVanVuuren_Microbian2/Karl/metadata_microbian2_26.02.2026.txt
ANCOM dir: False /vol/jlab/MicrobiomeAnalyses/Projects/JansenVanVuuren_Microbian2/Karl/real_ANCOMB_BC2
Output dir: /vol/jlab/MicrobiomeAnalyses/Projects/JansenVanVuuren_Microbian2/Karl/notebook_plots_2026_1


In [5]:
meta = pd.read_csv(META_FP, sep="\t", dtype=str)

display(meta.head())
print(meta.shape)
print(meta.columns.tolist())

,sample_name,altitude,approx_date_of_birth,breeding_facility,combination,combination_v2,combination_v3,country,date_of_arrival_at_gsi,date_of_collection,...,time_point,time_point_numeric,title,to_exclude,tumour_count,tumour_count_bin,Ymaze_percent_time_newarm,Ymaze_alternation_freq,OFT_percent_time_centralzone,NOR_discrimination_index_combined_normalised
0,16320.zr22868.68V3V4.S68.R1,not applicable,13.05.2025,Jackson_Labs,Apc_sham,Apc_sham_male_1,Apc_sham_male,Germany,11.06.2025,17.06.2025,...,baseline_1,-7,microbian2,FALSCH,16,1,not applicable,not applicable,not applicable,not applicable
1,16320.zr22868.91V3V4.S95.R1,not applicable,13.05.2025,Jackson_Labs,Apc_sham,Apc_sham_male_2,Apc_sham_male,Germany,11.06.2025,20.06.2025,...,baseline_2,-4,microbian2,FALSCH,16,1,not applicable,not applicable,not applicable,not applicable
2,16320.zr22868.391V3V4.S407.R1,not applicable,13.05.2025,Jackson_Labs,Apc_sham,Apc_sham_male_1,Apc_sham_male,Germany,11.06.2025,17.06.2025,...,baseline_1,-7,microbian2,FALSCH,19,1,not applicable,not applicable,not applicable,not applicable
3,16320.zr22868.407V3V4.S427.R1,not applicable,13.05.2025,Jackson_Labs,Apc_sham,Apc_sham_male_2,Apc_sham_male,Germany,11.06.2025,20.06.2025,...,baseline_2,-4,microbian2,FALSCH,19,1,not applicable,not applicable,not applicable,not applicable
4,16320.zr22868.227V3V4.S235.R1,not applicable,13.05.2025,Jackson_Labs,Apc_sham,Apc_sham_male_1,Apc_sham_male,Germany,11.06.2025,17.06.2025,...,baseline_1,-7,microbian2,FALSCH,20,2,not applicable,not applicable,not applicable,not applicable


(551, 72)
['sample_name', 'altitude', 'approx_date_of_birth', 'breeding_facility', 'combination', 'combination_v2', 'combination_v3', 'country', 'date_of_arrival_at_gsi', 'date_of_collection', 'date_of_irradiation', 'date_of_weight_measurement', 'depth', 'description', 'description_of_treatment', 'elevation', 'empo_1', 'empo_2', 'empo_3', 'empo_4', 'env_biome', 'env_feature', 'env_material', 'env_package', 'experimental_group', 'fecal_collection_id', 'geo_loc_name', 'gsi_cage_allocation', 'host_age', 'host_age_units', 'host_body_habitat', 'host_body_mass_index', 'host_body_product', 'host_body_site', 'host_common_name', 'host_genotype_name', 'host_height', 'host_height_units', 'host_life_stage', 'host_scientific_name', 'host_subject_id', 'host_taxid', 'host_weight', 'host_weight_units', 'isolate_scientific_name', 'isolate_strain_taxonomy_id', 'latitude', 'longitude', 'mice_model', 'normal_food', 'normal_food_package_number', 'old_name', 'physical_specimen_location', 'physical_specimen_

In [6]:
TIMEPOINTS = [
    "baseline1",
    "baseline2",
    "baseline3",
    "day1",
    "day3",
    "day7",
    "day14",
]

TIMEPOINT_MAP = {
    "baseline_1": "baseline1",
    "baseline_2": "baseline2",
    "baseline_3": "baseline3",
    "baseline1": "baseline1",
    "baseline2": "baseline2",
    "baseline3": "baseline3",
    "day_1": "day1",
    "day_3": "day3",
    "day_7": "day7",
    "day_14": "day14",
    "day1": "day1",
    "day3": "day3",
    "day7": "day7",
    "day14": "day14",
}

COL_SAMPLE = "sample_name"
COL_TIME = "time_point"
COL_TREAT = "description_of_treatment"
COL_GENO = "mice_model"
COL_SEX = "sex"

meta[COL_TIME] = meta[COL_TIME].replace(TIMEPOINT_MAP)

for col in [COL_TIME, COL_TREAT, COL_GENO, COL_SEX]:
    print("\n", col)
    print(meta[col].value_counts(dropna=False))


 time_point
time_point
baseline1      79
baseline2      79
baseline3      79
day_1_post     79
day_3_post     79
day_7_post     79
day_14_post    77
Name: count, dtype: int64

 description_of_treatment
description_of_treatment
sham              273
irradiated        271
not applicable      7
Name: count, dtype: int64

 mice_model
mice_model
WT                273
Apc               271
not applicable      7
Name: count, dtype: int64

 sex
sex
female            273
male              271
not applicable      7
Name: count, dtype: int64


In [11]:
metadata_config = MetadataConfig(
    sample_col=COL_SAMPLE,
    timepoint_col=COL_TIME,
    comparison_col=COL_TREAT,
    timepoint_map=TIMEPOINT_MAP,
    timepoints=TIMEPOINTS,
    allowed_values={
        COL_TREAT: ["sham", "irradiated"],
        COL_GENO: ["WT", "Apc"],
        COL_SEX: ["female", "male"],
    },
)

comparison_config = ComparisonConfig(
    effect_col="description_of_treatment::sham",
    positive_label="irradiated",
    negative_label="sham",
    lfc_sign=-1,
)

path_config = PathConfig(
    ancom_export_dir=ANCOM_EXPORT_DIR,
    output_dir=OUTDIR,
)

heatmap_config = HeatmapConfig(
    metadata=metadata_config,
    comparison=comparison_config,
    paths=path_config,
    q_cutoff=0.05,
    min_sig_cells_per_taxon=1,
    split_after_timepoint="baseline3",
    cell_text_mode="relative_abundance",
)

TypeError: ComparisonConfig.__init__() got an unexpected keyword argument 'effect_col'